# 👁️ Smart Video Analytics — People Counting & Zone Intrusion (YOLO26, no training)

This project uses a **pretrained** YOLO26 model — nothing to train, no dataset to collect. YOLO26 already knows `person`, `car`, `bus`, `truck`, and 76 other COCO classes out of the box. We add **tracking** (persistent IDs across frames) and turn that into a real analytics system:

- **Live people count** in the scene
- **Line-crossing counter** — how many entered / exited (retail footfall, capacity)
- **Restricted-zone intrusion** — flag anyone inside a drawn area (security)

**Real applications:** retail footfall, crowd/queue management, occupancy limits, perimeter security.

> ⚙️ `Runtime → Change runtime type → T4 GPU` for real-time speed (works on CPU too, just slower).


## 1. Setup

In [ ]:
!pip install -q ultralytics
import ultralytics; ultralytics.checks()

In [ ]:
from ultralytics import YOLO
# Pretrained on COCO — downloads automatically, NO training needed.
model = YOLO("yolo26n.pt")     # n=nano (fast). Use yolo26s/m.pt for more accuracy.
print("Classes available:", len(model.names))
print("person is class id:", [k for k,v in model.names.items() if v=='person'])

## 2. Get a test video

Any video with people works. Grab a free one from Pexels (e.g. a street or mall clip),
upload it via the Files panel, and set the path. Or use the sample below.


In [ ]:
import os
VIDEO = "/content/people.mp4"   # <-- upload a video and set this path
print("exists:", os.path.exists(VIDEO))

## 3. The core idea — tracking gives persistent IDs

`model.track(..., persist=True)` assigns each object a stable ID that survives across frames.
Counting **unique IDs** = accurate counts even on long videos. This one loop is the whole engine;
every fancy feature is a thin layer on top of it.


In [ ]:
from collections import defaultdict

unique = defaultdict(set)
for r in model.track(VIDEO, stream=True, persist=True, tracker="bytetrack.yaml",
                     classes=[0], conf=0.3, verbose=False):   # classes=[0] -> people only
    for box in r.boxes:
        if box.id is None:
            continue
        unique[r.names[int(box.cls)]].add(int(box.id))

for cls, ids in unique.items():
    print(f"{cls}: {len(ids)} unique across the whole video")

## 4. Live people count + annotated video

Now we render an output video where each person is boxed with their ID, and the current
count is drawn on every frame.


In [ ]:
import cv2, numpy as np
from IPython.display import Video, display

cap = cv2.VideoCapture(VIDEO)
w  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps= cap.get(cv2.CAP_PROP_FPS) or 25
cap.release()

writer = cv2.VideoWriter("/content/counted.mp4",
                         cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

for r in model.track(VIDEO, stream=True, persist=True, tracker="bytetrack.yaml",
                     classes=[0], conf=0.3, verbose=False):
    frame = r.plot()                      # draws boxes + IDs
    n = 0 if r.boxes.id is None else len(r.boxes.id)
    cv2.rectangle(frame, (0,0), (330,60), (0,0,0), -1)
    cv2.putText(frame, f"People in frame: {n}", (15,40),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,255,120), 2)
    writer.write(frame)
writer.release()

# browser-friendly re-encode
!ffmpeg -y -loglevel error -i /content/counted.mp4 -vcodec libx264 /content/counted_h264.mp4
display(Video("/content/counted_h264.mp4", embed=True, width=720))

## 5. Line-crossing counter (in / out)

Ultralytics ships an `ObjectCounter` solution: define a line, and it tallies objects
crossing it cumulatively — exactly what footfall / capacity systems do.


In [ ]:
from ultralytics import solutions

# A horizontal line across the middle; tweak points to your scene.
line = [(0, h//2), (w, h//2)]

counter = solutions.ObjectCounter(
    model="yolo26n.pt",
    region=line,
    classes=[0],          # count people
    show=False,
)

cap = cv2.VideoCapture(VIDEO)
writer = cv2.VideoWriter("/content/line.mp4",
                         cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
while cap.isOpened():
    ok, frame = cap.read()
    if not ok: break
    out = counter(frame)          # returns annotated frame + counts
    writer.write(out.plot_im)
cap.release(); writer.release()

!ffmpeg -y -loglevel error -i /content/line.mp4 -vcodec libx264 /content/line_h264.mp4
display(Video("/content/line_h264.mp4", embed=True, width=720))

## 6. Restricted-zone intrusion (security)

Draw a polygon; anyone whose box-center enters it is flagged. This is the perimeter-security
use case. We use `RegionCounter`-style logic manually so you see how it works.


In [ ]:
from shapely.geometry import Point, Polygon

# Define a restricted zone (a rectangle in the left third here — edit to your scene).
zone_pts = [(int(w*0.05), int(h*0.15)), (int(w*0.40), int(h*0.15)),
            (int(w*0.40), int(h*0.95)), (int(w*0.05), int(h*0.95))]
zone = Polygon(zone_pts)

cap = cv2.VideoCapture(VIDEO)
writer = cv2.VideoWriter("/content/zone.mp4",
                         cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

for r in model.track(VIDEO, stream=True, persist=True, tracker="bytetrack.yaml",
                     classes=[0], conf=0.3, verbose=False):
    frame = r.plot()
    intruders = 0
    if r.boxes.id is not None:
        for box in r.boxes.xyxy.cpu().numpy():
            cx, cy = (box[0]+box[2])/2, (box[1]+box[3])/2
            if zone.contains(Point(cx, cy)):
                intruders += 1
    # draw the zone (red if breached, green if clear)
    color = (0,0,255) if intruders else (0,200,0)
    cv2.polylines(frame, [np.array(zone_pts)], True, color, 3)
    cv2.rectangle(frame, (0,0), (360,60), (0,0,0), -1)
    label = f"INTRUDERS: {intruders}" if intruders else "ZONE CLEAR"
    cv2.putText(frame, label, (15,40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 2)
    writer.write(frame)
cap.release(); writer.release()

!ffmpeg -y -loglevel error -i /content/zone.mp4 -vcodec libx264 /content/zone_h264.mp4
display(Video("/content/zone_h264.mp4", embed=True, width=720))

## 7. Recap

- Loaded **pretrained** YOLO26 — zero training.
- Turned detection → tracking (persistent IDs) → three real analytics features.
- Each feature is a thin layer over the same `model.track(...)` loop.

**Next:** the accompanying Flask app wraps all of this behind an impressive web dashboard —
upload a video, pick a mode (count / line / zone), and watch the annotated result with live stats.

**Swap the use case for free:** change `classes=[0]` (person) to `[2]` (car) for traffic counting,
`[2,3,5,7]` for mixed vehicles, etc. Same code, new application.
